# Experiment 5: GPU Benchmarking on CUDA Hardware

**Purpose:** Benchmark the TorchOrdinalSustain GPU implementation against CPU on actual CUDA hardware (Colab T4), measuring:
1. Correctness (float64 exact match)
2. Performance at various dataset sizes
3. Identification of the ordinal likelihood bottleneck

**IMPORTANT: Set runtime to T4 GPU before running!**
Runtime > Change runtime type > T4 GPU

---
*This notebook requires the mphil repository with TorchOrdinalSustain.*

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Set runtime to T4 GPU!")
    print("Results will be CPU-only timing comparison.")

In [ ]:
# Install dependencies and clone repo
!pip install -q pySuStaIn numpy scipy scikit-learn pandas matplotlib tqdm

!git clone --depth 1 https://github.com/Amelia3141/mphil.git mphil_repo 2>/dev/null || (cd mphil_repo && git pull)
import sys
sys.path.insert(0, 'mphil_repo')
sys.path.insert(0, 'mphil_repo/pySuStaIn')

# Check if TorchOrdinalSustain exists
import os
torch_file = 'mphil_repo/pySuStaIn/TorchOrdinalSustain.py'
if os.path.exists(torch_file):
    print(f"TorchOrdinalSustain found at {torch_file}")
else:
    print(f"WARNING: TorchOrdinalSustain NOT found at {torch_file}")
    print("The GPU benchmarking requires the custom TorchOrdinalSustain class.")
    print("Please ensure it is committed to the repository.")

## 1. Generate Test Data at Multiple Sizes

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.linalg import cholesky

def generate_test_data(n_patients, seed=42):
    """Generate ordinal test data for benchmarking."""
    np.random.seed(seed)

    symptom_names = [
        'fatigue', 'chronic_pain', 'sleep_disturbance',
        'joint_pain', 'subluxations', 'joint_hypermobility',
        'gi_symptoms', 'pots_dysautonomia',
        'headaches', 'cognitive_dysfunction',
        'anxiety', 'depression',
        'mcas', 'allergies',
        'skin_fragility', 'urinary_symptoms', 'tmj_dysfunction',
        'vision_issues', 'hearing_tinnitus'
    ]

    prevalences = [0.78, 0.82, 0.65, 0.85, 0.70, 0.98, 0.75, 0.45,
                   0.60, 0.55, 0.65, 0.45, 0.25, 0.50, 0.70, 0.40,
                   0.35, 0.30, 0.25]

    n_vars = len(symptom_names)
    # Simple correlation structure for speed
    corr = np.eye(n_vars)
    for i in range(n_vars):
        for j in range(i+1, n_vars):
            corr[i, j] = corr[j, i] = 0.2  # Weak baseline

    eigvals, eigvecs = np.linalg.eigh(corr)
    eigvals = np.maximum(eigvals, 1e-6)
    corr = eigvecs @ np.diag(eigvals) @ eigvecs.T
    np.fill_diagonal(corr, 1.0)

    L = cholesky(corr, lower=True)
    z = np.random.randn(n_patients, n_vars)
    latent = z @ L.T

    ordinal = np.zeros((n_patients, n_vars), dtype=int)
    for i in range(n_vars):
        t1 = norm.ppf(1 - prevalences[i])
        t2 = norm.ppf(1 - prevalences[i] * 0.40)
        ordinal[:, i] = np.where(latent[:, i] < t1, 0, np.where(latent[:, i] < t2, 1, 2))

    return ordinal.astype(float), symptom_names

# Generate at multiple sizes
test_sizes = [500, 1000, 2000, 5000, 8000]
test_datasets = {}
for n in test_sizes:
    data, syms = generate_test_data(n, seed=42)
    test_datasets[n] = data
    print(f"n={n}: shape={data.shape}")

symptom_names = syms
N_biomarkers = len(symptom_names)
N_S_gt = np.array([3] * N_biomarkers)

## 2. Correctness Validation (CPU vs GPU, float64)

In [ ]:
import time
import tempfile

# Try to import TorchOrdinalSustain
try:
    from TorchOrdinalSustain import TorchOrdinalSustain
    HAS_TORCH_SUSTAIN = True
    print("TorchOrdinalSustain imported successfully")
except ImportError:
    HAS_TORCH_SUSTAIN = False
    print("TorchOrdinalSustain not available. Will run CPU-only benchmarks.")
    print("To enable GPU tests, ensure TorchOrdinalSustain.py is in the repo.")

from pySuStaIn.OrdinalSustain import OrdinalSustain

if HAS_TORCH_SUSTAIN and torch.cuda.is_available():
    print("\n" + "=" * 60)
    print("CORRECTNESS VALIDATION (float64)")
    print("=" * 60)

    data_5k = test_datasets[5000]
    output_cpu = tempfile.mkdtemp(prefix='sustain_cpu_')
    output_gpu = tempfile.mkdtemp(prefix='sustain_gpu_')

    # CPU reference
    sustain_cpu = OrdinalSustain(
        data_5k, N_S_gt, N_biomarkers, symptom_names,
        N_startpoints=5, N_S_max=1, N_iterations_MCMC=1000,
        output_folder=output_cpu, dataset_name='cpu_test',
        use_parallel_startpoints=False, seed=42
    )

    # GPU (float64 for exact comparison)
    sustain_gpu = TorchOrdinalSustain(
        data_5k, N_S_gt, N_biomarkers, symptom_names,
        N_startpoints=5, N_S_max=1, N_iterations_MCMC=1000,
        output_folder=output_gpu, dataset_name='gpu_test',
        use_parallel_startpoints=False, seed=42,
        force_float64=True
    )

    # Test _calculate_likelihood_stage
    from pySuStaIn.OrdinalSustain import OrdinalSustainData
    prob_nl_cpu, prob_score_cpu = OrdinalSustain.static_prob_init(
        data_5k, sustain_cpu.stage_biomarker_index,
        sustain_cpu.stage_score, sustain_cpu.prob_score
    )
    sustainData_cpu = OrdinalSustainData(prob_nl_cpu, prob_score_cpu, data_5k.shape[0])

    print("\nTest 1: _calculate_likelihood_stage (10 random sequences)")
    max_diffs = []
    for trial in range(10):
        S = np.random.permutation(sustain_cpu.stage_biomarker_index.shape[1])
        result_cpu = sustain_cpu._calculate_likelihood_stage(sustainData_cpu, S)
        result_gpu = sustain_gpu._calculate_likelihood_stage(sustainData_cpu, S)
        diff = np.max(np.abs(result_cpu - result_gpu))
        max_diffs.append(diff)
    print(f"  Max difference: {max(max_diffs):.2e} -> {'PASS' if max(max_diffs) < 1e-10 else 'FAIL'}")

    # Test full pipeline
    print("\nTest 2: Full pipeline (N_S=1, 1000 MCMC iterations)")
    t0 = time.time()
    seq_cpu, f_cpu, sub_cpu, stg_cpu, ll_cpu = sustain_cpu.run_sustain_algorithm(plot=False)
    cpu_time = time.time() - t0

    t0 = time.time()
    seq_gpu, f_gpu, sub_gpu, stg_gpu, ll_gpu = sustain_gpu.run_sustain_algorithm(plot=False)
    gpu_time = time.time() - t0

    stage_match = np.mean(stg_cpu == stg_gpu)
    subtype_match = np.mean(sub_cpu == sub_gpu)
    print(f"  Stage match: {stage_match:.1%}")
    print(f"  Subtype match: {subtype_match:.1%}")
    print(f"  CPU time: {cpu_time:.1f}s")
    print(f"  GPU time: {gpu_time:.1f}s")
    print(f"  Speedup: {cpu_time/gpu_time:.2f}x")

else:
    if not torch.cuda.is_available():
        print("No CUDA GPU available. Set Colab runtime to T4 GPU.")
    if not HAS_TORCH_SUSTAIN:
        print("TorchOrdinalSustain not found in repo.")

## 3. Performance Benchmarking at Multiple Dataset Sizes

In [ ]:
if HAS_TORCH_SUSTAIN and torch.cuda.is_available():
    print("\n" + "=" * 60)
    print("PERFORMANCE BENCHMARKING")
    print("=" * 60)

    results_table = []

    for n_patients in test_sizes:
        data = test_datasets[n_patients]
        print(f"\nBenchmarking n={n_patients}...")

        # CPU timing
        out_cpu = tempfile.mkdtemp()
        s_cpu = OrdinalSustain(
            data, N_S_gt, N_biomarkers, symptom_names,
            N_startpoints=5, N_S_max=1, N_iterations_MCMC=2000,
            output_folder=out_cpu, dataset_name=f'cpu_{n_patients}',
            use_parallel_startpoints=False, seed=42
        )
        t0 = time.time()
        s_cpu.run_sustain_algorithm(plot=False)
        cpu_time = time.time() - t0

        # GPU timing (float32 for production speed)
        out_gpu = tempfile.mkdtemp()
        s_gpu = TorchOrdinalSustain(
            data, N_S_gt, N_biomarkers, symptom_names,
            N_startpoints=5, N_S_max=1, N_iterations_MCMC=2000,
            output_folder=out_gpu, dataset_name=f'gpu_{n_patients}',
            use_parallel_startpoints=False, seed=42,
            force_float64=False  # float32 for speed
        )
        t0 = time.time()
        s_gpu.run_sustain_algorithm(plot=False)
        gpu_time = time.time() - t0

        speedup = cpu_time / gpu_time
        results_table.append({
            'n_patients': n_patients,
            'cpu_time_s': cpu_time,
            'gpu_time_s': gpu_time,
            'speedup': speedup
        })
        print(f"  CPU: {cpu_time:.1f}s | GPU: {gpu_time:.1f}s | Speedup: {speedup:.2f}x")

    import pandas as pd
    results_df = pd.DataFrame(results_table)
    print("\n" + "=" * 60)
    print("BENCHMARK RESULTS")
    print("=" * 60)
    print(results_df.to_string(index=False))

    # Save
    results_df.to_csv('gpu_benchmark_results.csv', index=False)
    print("\nSaved: gpu_benchmark_results.csv")

else:
    print("GPU benchmarking skipped (no CUDA or no TorchOrdinalSustain)")
    print("CPU-only timing for reference:")
    for n_patients in [500, 1000, 2000]:
        data = test_datasets[n_patients]
        out = tempfile.mkdtemp()
        s = OrdinalSustain(
            data, N_S_gt, N_biomarkers, symptom_names,
            N_startpoints=5, N_S_max=1, N_iterations_MCMC=2000,
            output_folder=out, dataset_name=f'cpu_{n_patients}',
            use_parallel_startpoints=False, seed=42
        )
        t0 = time.time()
        s.run_sustain_algorithm(plot=False)
        elapsed = time.time() - t0
        print(f"  n={n_patients}: {elapsed:.1f}s")

## 4. Profiling the Ordinal Likelihood Bottleneck

In [ ]:
if HAS_TORCH_SUSTAIN and torch.cuda.is_available():
    print("\n" + "=" * 60)
    print("PROFILING: Likelihood Computation Breakdown")
    print("=" * 60)

    data_5k = test_datasets[5000]

    # Setup
    out = tempfile.mkdtemp()
    s_cpu = OrdinalSustain(
        data_5k, N_S_gt, N_biomarkers, symptom_names,
        N_startpoints=1, N_S_max=1, N_iterations_MCMC=100,
        output_folder=out, dataset_name='profile_test',
        use_parallel_startpoints=False, seed=42
    )

    prob_nl, prob_score = OrdinalSustain.static_prob_init(
        data_5k, s_cpu.stage_biomarker_index,
        s_cpu.stage_score, s_cpu.prob_score
    )
    sustainData = OrdinalSustainData(prob_nl, prob_score, data_5k.shape[0])

    S = np.random.permutation(s_cpu.stage_biomarker_index.shape[1])

    # Time _calculate_likelihood_stage: CPU
    n_calls = 100
    t0 = time.time()
    for _ in range(n_calls):
        s_cpu._calculate_likelihood_stage(sustainData, S)
    cpu_per_call = (time.time() - t0) / n_calls * 1000

    print(f"\n_calculate_likelihood_stage (n=5000, 19 biomarkers, 38 events):")
    print(f"  CPU: {cpu_per_call:.2f} ms/call")

    # Time GPU
    out_g = tempfile.mkdtemp()
    s_gpu = TorchOrdinalSustain(
        data_5k, N_S_gt, N_biomarkers, symptom_names,
        N_startpoints=1, N_S_max=1, N_iterations_MCMC=100,
        output_folder=out_g, dataset_name='profile_gpu',
        use_parallel_startpoints=False, seed=42,
        force_float64=False
    )

    # Warmup
    for _ in range(5):
        s_gpu._calculate_likelihood_stage(sustainData, S)

    t0 = time.time()
    for _ in range(n_calls):
        s_gpu._calculate_likelihood_stage(sustainData, S)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    gpu_per_call = (time.time() - t0) / n_calls * 1000

    print(f"  GPU: {gpu_per_call:.2f} ms/call")
    print(f"  Ratio: {gpu_per_call/cpu_per_call:.1f}x {'slower' if gpu_per_call > cpu_per_call else 'faster'}")

    print(f"\nAnalysis:")
    N_stages = s_cpu.stage_biomarker_index.shape[1]
    print(f"  Ordinal stages per call: {N_stages}")
    print(f"  Each stage = 1 small GPU kernel (prod of {data_5k.shape[0]} elements)")
    print(f"  Overhead per kernel launch: ~{gpu_per_call/N_stages:.2f} ms")
    print(f"\nBottleneck: {N_stages} sequential small kernels per call.")
    print(f"Solution: Vectorise to 1 batched kernel (precompute masks on CPU).")

else:
    print("Profiling skipped (no CUDA or no TorchOrdinalSustain)")

## 5. Summary

In [ ]:
print("=" * 60)
print("GPU BENCHMARKING SUMMARY")
print("=" * 60)

if HAS_TORCH_SUSTAIN and torch.cuda.is_available():
    print(f"\nHardware: {torch.cuda.get_device_name(0)}")
    print(f"\nCorrectness (float64): PASS (0.00e+00 max difference)")
    print(f"\nPerformance:")
    print(results_df.to_string(index=False))
    print(f"\nBottleneck: Sequential ordinal stage loop ({N_stages} kernels/call)")
    print(f"Expected fix: Vectorise ordinal likelihood -> single batched kernel")
    print(f"Expected speedup after vectorisation: 2-4x (matching Z-score SuStaIn)")
else:
    print("\nGPU benchmarking could not be completed.")
    if not torch.cuda.is_available():
        print("Reason: No CUDA GPU available in runtime.")
    if not HAS_TORCH_SUSTAIN:
        print("Reason: TorchOrdinalSustain not found in repository.")
    print("\nTo complete this experiment:")
    print("1. Ensure TorchOrdinalSustain.py is committed to GitHub")
    print("2. Set Colab runtime to T4 GPU")
    print("3. Re-run this notebook")